In [1]:
import pandas as pd
import numpy as np
import re
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [2]:
# Cargamos los datos y mostramos las columnas
df = pd.read_csv("datos.csv")

In [3]:
########################################### Funciones ###########################################

def clean_provincia(s):
    if pd.isna(s):
        return s
    s = str(s)
    s = s.replace("\xa0", " ")
    s = re.sub(r"\[[^\]]*\]", "", s)   
    s = re.sub(r"\s+", " ", s).strip()
    return s

def unificar_columnas(df, nombre_base):
    cols = [c for c in df.columns if c == nombre_base or c.startswith(nombre_base + "_")]
    if not cols:
        return df
    
    df[nombre_base] = df[cols].bfill(axis=1).iloc[:, 0]
    cols_a_borrar = [c for c in cols if c != nombre_base]
    df = df.drop(columns=cols_a_borrar, errors="ignore")
    return df

def clean_column_name(col):
    col = str(col).strip()

    # quitar referencias tipo [2], [3], [a]
    col = re.sub(r"\[[^\]]*\]", "", col)

    # quitar prefijos/sufijos de pandas tipo .1
    col = re.sub(r"\.\d+$", "", col)

    # quitar columnas con numeros añadidos
    col = re.sub(r"_\d+$", "", col)

    # quitar espacios duplicados
    col = re.sub(r"\s+", " ", col).strip()

    return col

def limpiar_numero(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()

    x = x.replace("\xa0", " ")
    x = x.replace("%", "")
    x = x.replace("€", "")
    x = x.replace("—", "")

    x = x.replace(" ", "")

    # solo eliminar separadores de miles si hay patrón claro
    if x.count(",") == 1 and x.count(".") > 1:
        x = x.replace(".", "")
    elif x.count(".") == 1 and x.count(",") > 1:
        x = x.replace(",", "")

    # caso típico España: decimal con coma
    if "," in x and "." not in x:
        x = x.replace(",", ".")

    return pd.to_numeric(x, errors="coerce")

def clean_text_value(x):
    if pd.isna(x):
        return x
    x = str(x)
    x = x.replace("\xa0", " ")   # espacio no separable
    x = x.replace("\u200b", "")  # zero-width space
    x = x.strip()
    return x

In [4]:
# Eliminar filas y columnas que no interesan
del_rows = ["España", "Total", "Otras (Chafarinas y Vélez de la Gomera)"]
df = df[~df["Provincia"].isin(del_rows)] 
df = df.loc[:, ~df.columns.str.contains(r"^Unnamed", na=False)]


columnas_a_eliminar = [
    c for c in df.columns
    if c.startswith("Unnamed")
    or c.startswith("Escudo")
    or c.startswith("Mapa")
    or c.startswith("Posición")
    or c.startswith("Puesto")
]
df = df.drop(columns=columnas_a_eliminar, errors="ignore")


# Cambiamos los nombres a las columnas
df.columns = [clean_column_name(c) for c in df.columns]

# Limpieza del texto de wikipedia
df = df.map(clean_text_value)

# Normalizar nombres de provincias
df["Provincia"] = df["Provincia"].apply(clean_provincia)


# Convertir números que están como texto
num_cols = df.select_dtypes(include=["number"]).columns

for col in num_cols:
    df[col] = df[col].apply(limpiar_numero)

# Unificar columnas
df = unificar_columnas(df, "Comunidad autónoma")
df = unificar_columnas(df, "Capital")
df = unificar_columnas(df, "Población")
df = unificar_columnas(df, "Provincia")

# Añadimos el prefijo PIB a las columnas de 2016, 2017...
lst_pib = ['2016', '2017', '2018', '2019','2020', '2021', '2022', '2023']
dct_col = {col : 'PIB_'+col for col in lst_pib}
df.rename(columns=dct_col, inplace=True)

In [5]:
# Corregimos el desorden de la columna provincia.1 y % de extranjeros
df_ext = df[['Provincia.1', "% de extranjeros"]]
df.drop(columns=['Provincia.1', "% de extranjeros"], inplace=True)

df_ext.rename(columns={'Provincia.1':'Provincia'}, inplace=True)

df = df.merge(df_ext, on='Provincia', how='left')

In [6]:
df.head(5)

,Provincia,Superficie (km²),Longitud de costa (km)​​,Código postal,Código Ministerio del Interior,Extranjeros totales,Población,Presupuesto (€),Porcentaje población,Densidad (hab./km²),Superficie (km²)​,Porcentaje superficie,Capital,Ciudad más poblada*,Comunidad autónoma,PIB_2016,PIB_2017,PIB_2018,PIB_2019,PIB_2020,PIB_2021,PIB_2022,PIB_2023,TCAC 2016-23,Órgano de gobierno y administración,Sede (ciudad),Comunidad autónoma,% de extranjeros
0,Albacete,NaN,NaN,2.0,AB,32.439,390 751,179 014 529,0.8 %,26.18,14 924,2.95 %,Albacete,Albacete,Castilla-La Mancha,7.570.409,7.992.903,8.285.269,8.627.212,8.010.434,8.853.382,9.526.877,10.285.196,"35,86%",Diputación de Albacete,Palacio Provincial de Albacete (Albacete),Castilla-La Mancha,84.0
1,Alicante,5816,244,3.0,A,456.257,2 033 566,313 871 224,4.14 %,349.59,5817,1.15 %,Alicante,Alicante,Comunidad Valenciana,33.899.084,35.422.930,36.463.569,37.807.882,34.503.374,38.470.028,41.939.432,46.472.807,"37,09%",Diputación de Alicante,Palacio Provincial de Alicante (Alicante),Comunidad Valenciana,230.0
2,Almería,8774,249,4.0,AL,168.886,770 554,225 750 000,1.57 %,87.81,8775,1.73 %,Almería,Almería,Andalucía,14.185.706,15.159.388,15.282.092,16.003.582,14.764.342,14.861.150,16.494.994,18.250.223,"28,65%",Diputación de Almería,Edificio de la Diputación Provincial de Almerí...,Andalucía,223.0
3,Asturias,10 603,401,33.0,O,58.387,1 015 128,291 900 000,2.07 %,95.73,10 604,2.1 %,Oviedo,Gijón,Asturias,21.675.441,22.538.267,23.185.680,23.666.618,21.376.392,23.617.641,26.690.102,28.350.371,"30,79%",No tiene[1],No tiene[1],Asturias,59.0
4,Badajoz,NaN,NaN,6.0,BA,25.469,665 155,122 565 847,1.35 %,30.56,21 766,4.3 %,Badajoz,Badajoz,Extremadura,11.776.505,12.419.839,12.609.609,12.838.980,11.979.224,12.729.424,13.644.123,14.927.003,"26,75%",Diputación de Badajoz,NaN,Extremadura,38.0


In [ ]:
# import pandas as pd
# import numpy as np

# df = pd.read_csv("datos_2.csv")

# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)


# def normalizar_provincia(x):
#     if pd.isna(x):
#         return x

#     x = str(x).strip()

#     equivalencias = {
#         "Coruña, La": "La Coruña",
#         "Palmas, Las": "Las Palmas",
#         "Rioja, La": "La Rioja",
#         "Baleares": "Islas Baleares",
#         "Gerona": "Girona",
#         "Lérida": "Lleida",
#         "Orense": "Ourense",
#         "Álava": "Araba/Álava",
#         "Guipúzcoa": "Gipuzkoa",
#         "Vizcaya": "Bizkaia"
#     }

#     return equivalencias.get(x, x)


# # 1. eliminar filas que no interesan
# filas_basura = ["España", "Total", "Otras (Chafarinas y Vélez de la Gomera)", "-", "nan"]
# df = df[~df["Provincia"].astype(str).isin(filas_basura)].copy()

# # 2. normalizar nombres de provincias
# df["Provincia"] = df["Provincia"].astype(str).str.strip().apply(normalizar_provincia)


# # 3. función para unificar columnas repetidas
# def unificar_columnas(df, nombre_base):
#     cols = [c for c in df.columns if c == nombre_base or c.startswith(nombre_base + "_")]
#     if not cols:
#         return df

#     df[nombre_base] = df[cols].bfill(axis=1).iloc[:, 0]
#     cols_a_borrar = [c for c in cols if c != nombre_base]
#     df = df.drop(columns=cols_a_borrar, errors="ignore")
#     return df


# # 4. unificar columnas equivalentes
# bases_a_unificar = [
#     "Comunidad autónoma",
#     "Capital",
#     "Ciudad más poblada*",
#     "Población",
#     "Presupuesto (€)",
#     "Densidad (hab./km²)",
#     "Superficie (km²)",
#     "Código postal",
#     "Código Ministerio del Interior",
#     "Órgano de gobierno y administración",
#     "Sede (ciudad)",
#     "Extranjeros totales",
#     "% de extranjeros",
#     "2016",
#     "2017",
#     "2018",
#     "2019",
#     "2020",
#     "2021",
#     "2022",
#     "2023",
#     "TCAC 2016-23"
# ]

# for base in bases_a_unificar:
#     df = unificar_columnas(df, base)


# # 5. consolidar filas duplicadas por provincia
# def first_non_null(series):
#     s = series.dropna()
#     return s.iloc[0] if not s.empty else np.nan

# df = df.groupby("Provincia", as_index=False).agg(first_non_null)


# # 6. limpieza numérica
# def limpiar_numero(x):
#     if pd.isna(x):
#         return np.nan

#     x = str(x).strip()

#     # nulos y símbolos raros
#     if x in ["-", "—", "", "nan", "None"]:
#         return np.nan

#     # quitar referencias tipo [2], [3], etc.
#     import re
#     x = re.sub(r"\[\d+\]", "", x)

#     # quitar espacios
#     x = x.replace(" ", "")

#     # quitar %
#     x = x.replace("%", "")

#     # formato español:
#     # 1.234.567,89 -> 1234567.89
#     # 390 751 -> 390751
#     x = x.replace(".", "")
#     x = x.replace(",", ".")

#     return pd.to_numeric(x, errors="coerce")


# # 7. columnas numéricas a convertir
# cols_numericas = [
#     "Extranjeros totales",
#     "% de extranjeros",
#     "Población",
#     "Presupuesto (€)",
#     "Densidad (hab./km²)",
#     "Superficie (km²)",
#     "2016",
#     "2017",
#     "2018",
#     "2019",
#     "2020",
#     "2021",
#     "2022",
#     "2023",
#     "TCAC 2016-23"
# ]

# for col in cols_numericas:
#     if col in df.columns:
#         df[col] = df[col].apply(limpiar_numero)


# # 8. ver columnas finales
# print(df.columns.tolist())

# # 9. guardar limpio
# df.to_csv("datos_limpios.csv", index=False)

# df

FileNotFoundError: [Errno 2] No such file or directory: 'datos_2.csv'

In [ ]:
# import pandas as pd
# import numpy as np
# import re

# df = pd.read_csv("datos_2.csv")

# pd.set_option("display.max_rows", None)
# pd.set_option("display.max_columns", None)


# # ----------------------------
# # 1. Normalizar provincias
# # ----------------------------
# def normalizar_provincia(x):
#     if pd.isna(x):
#         return x

#     x = str(x).strip()

#     equivalencias = {
#         "Coruña, La": "La Coruña",
#         "Palmas, Las": "Las Palmas",
#         "Rioja, La": "La Rioja",
#         "Baleares": "Islas Baleares",
#         "Gerona": "Girona",
#         "Lérida": "Lleida",
#         "Orense": "Ourense",
#         "Álava": "Araba/Álava",
#         "Guipúzcoa": "Gipuzkoa",
#         "Vizcaya": "Bizkaia"
#     }

#     return equivalencias.get(x, x)


# df["Provincia"] = df["Provincia"].astype(str).str.strip().apply(normalizar_provincia)

# filas_basura = ["España", "Total", "Otras (Chafarinas y Vélez de la Gomera)", "-", "nan"]
# df = df[~df["Provincia"].isin(filas_basura)].copy()


# # ----------------------------
# # 2. Eliminar columnas basura
# # ----------------------------
# # elimina columnas auxiliares que no quieres analizar
# columnas_a_eliminar = [
#     c for c in df.columns
#     if c.startswith("Unnamed")
#     or c.startswith("Escudo")
#     or c.startswith("Mapa")
#     or c.startswith("Posición")
#     or c.startswith("Puesto")
# ]

# # elimina provincias secundarias tipo Provincia.1_3, Provincia.2_...
# columnas_a_eliminar += [
#     c for c in df.columns
#     if c.startswith("Provincia.")   # Provincia.1_3, etc.
# ]

# df = df.drop(columns=columnas_a_eliminar, errors="ignore")


# # ----------------------------
# # 3. Unificar columnas repetidas
# # ----------------------------
# def unificar_columnas(df, nombre_base):
#     cols = [c for c in df.columns if c == nombre_base or c.startswith(nombre_base + "_")]
#     if not cols:
#         return df

#     # primer valor no nulo por fila
#     df[nombre_base] = df[cols].bfill(axis=1).iloc[:, 0]

#     # borrar duplicadas con sufijo
#     cols_a_borrar = [c for c in cols if c != nombre_base]
#     df = df.drop(columns=cols_a_borrar, errors="ignore")

#     return df


# bases_a_unificar = [
#     "Comunidad autónoma",
#     "Capital",
#     "Ciudad más poblada*",
#     "Población",
#     "Presupuesto (€)",
#     "Densidad (hab./km²)",
#     "Superficie (km²)",
#     "Código postal",
#     "Código Ministerio del Interior",
#     "Órgano de gobierno y administración",
#     "Sede (ciudad)",
#     "Extranjeros totales",
#     "% de extranjeros",
#     "2016",
#     "2017",
#     "2018",
#     "2019",
#     "2020",
#     "2021",
#     "2022",
#     "2023",
#     "TCAC 2016-23",
#     "Porcentaje población",
#     "Porcentaje superficie",
#     "Longitud de costa (km)[2]​[3]​"
# ]

# for base in bases_a_unificar:
#     df = unificar_columnas(df, base)


# # ----------------------------
# # 4. Consolidar filas duplicadas por provincia
# # ----------------------------
# def first_non_null(series):
#     s = series.dropna()
#     return s.iloc[0] if not s.empty else np.nan

# df = df.groupby("Provincia", as_index=False).agg(first_non_null)


# # ----------------------------
# # 5. Limpieza numérica mejorada
# # ----------------------------
# def limpiar_numero(x):
#     if pd.isna(x):
#         return np.nan

#     x = str(x).strip()

#     if x in ["", "-", "—", "nan", "None"]:
#         return np.nan

#     # quitar referencias [1], [2], etc.
#     x = re.sub(r"\[\d+\]", "", x).strip()

#     # quitar espacios duros y normales
#     x = x.replace("\xa0", " ")
#     x = x.replace(" ", "")

#     # quitar símbolo %
#     x = x.replace("%", "")

#     # casos:
#     # 1) 1.234.567,89 -> 1234567.89
#     if "," in x and "." in x:
#         x = x.replace(".", "")
#         x = x.replace(",", ".")
#         return pd.to_numeric(x, errors="coerce")

#     # 2) 7.570.409 -> entero con puntos de miles
#     if x.count(".") > 1:
#         x = x.replace(".", "")
#         return pd.to_numeric(x, errors="coerce")

#     # 3) 14 924 ya quedó como 14924 antes
#     # 4) 26.18 -> decimal correcto
#     # 5) 37,99 -> decimal con coma
#     if "," in x:
#         x = x.replace(",", ".")

#     return pd.to_numeric(x, errors="coerce")


# cols_numericas = [
#     "Longitud de costa (km)[2]​[3]​",
#     "Población",
#     "Presupuesto (€)",
#     "Porcentaje población",
#     "Densidad (hab./km²)",
#     "Superficie (km²)",
#     "Porcentaje superficie",
#     "Extranjeros totales",
#     "% de extranjeros",
#     "2016",
#     "2017",
#     "2018",
#     "2019",
#     "2020",
#     "2021",
#     "2022",
#     "2023",
#     "TCAC 2016-23"
# ]

# for col in cols_numericas:
#     if col in df.columns:
#         df[col] = df[col].apply(limpiar_numero)


# # ----------------------------
# # 6. Reordenar columnas útiles
# # ----------------------------
# columnas_preferidas = [
#     "Provincia",
#     "Comunidad autónoma",
#     "Capital",
#     "Ciudad más poblada*",
#     "Código postal",
#     "Código Ministerio del Interior",
#     "Superficie (km²)",
#     "Longitud de costa (km)[2]​[3]​",
#     "Población",
#     "Densidad (hab./km²)",
#     "Presupuesto (€)",
#     "Extranjeros totales",
#     "% de extranjeros",
#     "2016",
#     "2017",
#     "2018",
#     "2019",
#     "2020",
#     "2021",
#     "2022",
#     "2023",
#     "TCAC 2016-23",
#     "Órgano de gobierno y administración",
#     "Sede (ciudad)"
# ]

# columnas_finales = [c for c in columnas_preferidas if c in df.columns] + [
#     c for c in df.columns if c not in columnas_preferidas
# ]

# df = df[columnas_finales]


# # ----------------------------
# # 7. Guardar
# # ----------------------------
# df.to_csv("datos_limpios_2.csv", index=False)

# print(df.head())
# print("\nColumnas finales:")
# print(df.columns.tolist())
# print("\nShape:", df.shape)

     Provincia    Comunidad autónoma   Capital Ciudad más poblada*  \
0     Albacete    Castilla-La Mancha  Albacete            Albacete   
1     Alicante  Comunidad Valenciana  Alicante            Alicante   
2      Almería             Andalucía   Almería             Almería   
3  Araba/Álava            País Vasco   Vitoria             Vitoria   
4     Asturias              Asturias    Oviedo               Gijón   

   Código postal Código Ministerio del Interior  Superficie (km²)  \
0              2                             AB               NaN   
1              3                              A            5816.0   
2              4                             AL            8774.0   
3              1                             VI               NaN   
4             33                              O           10603.0   

   Longitud de costa (km)[2]​[3]​  Población  Densidad (hab./km²)  \
0                             NaN     390751                26.18   
1                         

In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("datos_limpios_2.csv")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)


# ----------------------------
# 1. Limpiar texto general
# ----------------------------
def limpiar_texto(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = re.sub(r"\[\d+\]", "", x)   # quita [1], [2], etc.
    x = x.replace("\xa0", " ")
    x = x.strip()
    if x in ["", "-", "—", "nan", "None"]:
        return np.nan
    return x

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(limpiar_texto)


# ----------------------------
# 2. Corregir Extranjeros totales
#    Aquí el punto es separador de miles
# ----------------------------
def limpiar_entero_miles(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = x.replace(" ", "")
    x = x.replace(".", "")
    x = x.replace(",", "")
    return pd.to_numeric(x, errors="coerce")

if "Extranjeros totales" in df.columns:
    df["Extranjeros totales"] = df["Extranjeros totales"].apply(limpiar_entero_miles)


# ----------------------------
# 3. Corregir columnas porcentuales/decimales
#    Aquí NO hay que quitar puntos decimales
# ----------------------------
def limpiar_decimal_suave(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    x = x.replace("%", "")
    x = x.replace("\xa0", "")
    x = x.replace(" ", "")
    x = x.replace(",", ".")
    return pd.to_numeric(x, errors="coerce")

for col in ["Densidad (hab./km²)", "% de extranjeros", "TCAC 2016-23", "Porcentaje población", "Porcentaje superficie"]:
    if col in df.columns:
        df[col] = df[col].apply(limpiar_decimal_suave)


# ----------------------------
# 4. Corregir columnas enteras que aún estén como texto
#    Pero sin tocar las que ya están bien
# ----------------------------
def convertir_si_texto_entero(col):
    if col in df.columns and df[col].dtype == "object":
        df[col] = (
            df[col]
            .astype(str)
            .str.replace("\xa0", "", regex=False)
            .str.replace(" ", "", regex=False)
            .str.replace(".", "", regex=False)
            .str.replace(",", "", regex=False)
        )
        df[col] = pd.to_numeric(df[col], errors="coerce")

for col in [
    "Población",
    "Presupuesto (€)",
    "2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023"
]:
    convertir_si_texto_entero(col)


# ----------------------------
# 5. Fusionar superficie auxiliar con la principal
# ----------------------------
col_aux = [c for c in df.columns if c.startswith("Superficie (km²)[2]")]
if col_aux:
    aux = col_aux[0]

    # convertir auxiliar si hace falta
    if df[aux].dtype == "object":
        df[aux] = (
            df[aux]
            .astype(str)
            .str.replace("\xa0", "", regex=False)
            .str.replace(" ", "", regex=False)
            .str.replace(".", "", regex=False)
            .str.replace(",", "", regex=False)
        )
        df[aux] = pd.to_numeric(df[aux], errors="coerce")

    # solo rellenar huecos de la principal
    if "Superficie (km²)" in df.columns:
        df["Superficie (km²)"] = df["Superficie (km²)"].fillna(df[aux])
        df = df.drop(columns=[aux])
    else:
        df = df.rename(columns={aux: "Superficie (km²)"})


# ----------------------------
# 6. Guardar versión corregida
# ----------------------------
df.to_csv("datos_final_bien.csv", index=False)

print(df.head())
print("\nShape:", df.shape)
print("\nColumnas:", df.columns.tolist())

     Provincia    Comunidad autónoma   Capital Ciudad más poblada*  \
0     Albacete    Castilla-La Mancha  Albacete            Albacete   
1     Alicante  Comunidad Valenciana  Alicante            Alicante   
2      Almería             Andalucía   Almería             Almería   
3  Araba/Álava            País Vasco   Vitoria             Vitoria   
4     Asturias              Asturias    Oviedo               Gijón   

   Código postal Código Ministerio del Interior Superficie (km²)  \
0              2                             AB           14 924   
1              3                              A           5816.0   
2              4                             AL           8774.0   
3              1                             VI             3037   
4             33                              O          10603.0   

   Longitud de costa (km)[2]​[3]​  Población  Densidad (hab./km²)  \
0                             NaN     390751                26.18   
1                           244.

In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("datos_raw.csv")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)


# ----------------------------
# 1. Normalizar provincia
# ----------------------------
def normalizar_provincia(x):
    if pd.isna(x):
        return x

    x = str(x).strip()

    equivalencias = {
        "Coruña, La": "La Coruña",
        "Palmas, Las": "Las Palmas",
        "Rioja, La": "La Rioja",
        "Baleares": "Islas Baleares",
        "Gerona": "Girona",
        "Lérida": "Lleida",
        "Orense": "Ourense",
        "Álava": "Araba/Álava",
        "Guipúzcoa": "Gipuzkoa",
        "Vizcaya": "Bizkaia"
    }

    return equivalencias.get(x, x)


df["Provincia"] = df["Provincia"].astype(str).str.strip().apply(normalizar_provincia)

filas_basura = ["España", "Total", "Otras (Chafarinas y Vélez de la Gomera)", "-", "nan"]
df = df[~df["Provincia"].isin(filas_basura)].copy()


# ----------------------------
# 2. Eliminar columnas claramente basura
# ----------------------------
cols_drop = [
    c for c in df.columns
    if c.startswith("Unnamed")
    or c.startswith("Escudo")
    or c.startswith("Mapa")
    or c.startswith("Posición")
    or c.startswith("Puesto")
    or c.startswith("Provincia.")
]

df = df.drop(columns=cols_drop, errors="ignore")


# ----------------------------
# 3. Unificar columnas equivalentes
# ----------------------------
def unificar_columnas(df, nombre_base):
    cols = [c for c in df.columns if c == nombre_base or c.startswith(nombre_base + "_")]
    if not cols:
        return df

    df[nombre_base] = df[cols].bfill(axis=1).iloc[:, 0]
    cols_a_borrar = [c for c in cols if c != nombre_base]
    df = df.drop(columns=cols_a_borrar, errors="ignore")
    return df


bases_a_unificar = [
    "Comunidad autónoma",
    "Capital",
    "Ciudad más poblada*",
    "Población",
    "Presupuesto (€)",
    "Densidad (hab./km²)",
    "Superficie (km²)",
    "Código postal",
    "Código Ministerio del Interior",
    "Órgano de gobierno y administración",
    "Sede (ciudad)",
    "Extranjeros totales",
    "% de extranjeros",
    "2016",
    "2017",
    "2018",
    "2019",
    "2020",
    "2021",
    "2022",
    "2023",
    "TCAC 2016-23",
    "Porcentaje población",
    "Porcentaje superficie",
    "Longitud de costa (km)[2]​[3]​"
]

for base in bases_a_unificar:
    df = unificar_columnas(df, base)


# ----------------------------
# 4. Consolidar duplicados por provincia
# ----------------------------
def first_non_null(series):
    s = series.dropna()
    return s.iloc[0] if not s.empty else np.nan

df = df.groupby("Provincia", as_index=False).agg(first_non_null)


# ----------------------------
# 5. Limpieza de texto
# ----------------------------
def limpiar_texto(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = re.sub(r"\[\d+\]", "", x)
    x = x.replace("\xa0", " ")
    x = x.strip()

    if x in ["", "-", "—", "nan", "None"]:
        return np.nan

    return x

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(limpiar_texto)


# ----------------------------
# 6. Funciones numéricas robustas
# ----------------------------
def numero_entero_es(x):
    """
    Enteros con formato español.
    Mantiene bien:
    - 32.439 -> 32439
    - 390 751 -> 390751
    - 1015499.0 -> 1015499
    """
    if pd.isna(x):
        return np.nan

    if isinstance(x, (int, np.integer)):
        return int(x)

    if isinstance(x, (float, np.floating)):
        if np.isnan(x):
            return np.nan
        return int(round(x))

    s = limpiar_texto(x)
    if pd.isna(s):
        return np.nan

    s = str(s).replace(" ", "")

    # si tiene varios puntos, son miles
    if s.count(".") > 1:
        s = s.replace(".", "")
        return pd.to_numeric(s, errors="coerce")

    # si tiene un punto y detrás hay exactamente 3 dígitos, probablemente miles
    if s.count(".") == 1 and "," not in s:
        izq, der = s.split(".")
        if der.isdigit() and len(der) == 3:
            s = izq + der
            return pd.to_numeric(s, errors="coerce")

    # si es tipo 1015499.0, convertir a int
    val = pd.to_numeric(s, errors="coerce")
    if pd.notna(val):
        return int(round(val))

    return np.nan


def numero_decimal_es(x):
    """
    Decimales con formato español.
    Mantiene bien:
    - 26.18 -> 26.18
    - 7,4 -> 7.4
    - 37,99 -> 37.99
    - 1.234,56 -> 1234.56
    """
    if pd.isna(x):
        return np.nan

    if isinstance(x, (int, np.integer, float, np.floating)):
        return float(x)

    s = limpiar_texto(x)
    if pd.isna(s):
        return np.nan

    s = str(s).replace(" ", "")

    if "," in s and "." in s:
        s = s.replace(".", "")
        s = s.replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")

    return pd.to_numeric(s, errors="coerce")


# ----------------------------
# 7. Convertir columnas por tipo
# ----------------------------
cols_enteras = [
    "Código postal",
    "Superficie (km²)",
    "Longitud de costa (km)[2]​[3]​",
    "Población",
    "Presupuesto (€)",
    "Extranjeros totales",
    "2016",
    "2017",
    "2018",
    "2019",
    "2020",
    "2021",
    "2022",
    "2023"
]

for col in cols_enteras:
    if col in df.columns:
        df[col] = df[col].apply(numero_entero_es)

cols_decimales = [
    "Densidad (hab./km²)",
    "% de extranjeros",
    "TCAC 2016-23",
    "Porcentaje población",
    "Porcentaje superficie"
]

for col in cols_decimales:
    if col in df.columns:
        df[col] = df[col].apply(numero_decimal_es)


# ----------------------------
# 8. Si queda una superficie auxiliar, usarla para rellenar
# ----------------------------
col_aux = [c for c in df.columns if c.startswith("Superficie (km²)[2]")]
if col_aux:
    aux = col_aux[0]
    df[aux] = df[aux].apply(numero_entero_es)

    if "Superficie (km²)" not in df.columns:
        df["Superficie (km²)"] = np.nan

    df["Superficie (km²)"] = df["Superficie (km²)"].fillna(df[aux])
    df = df.drop(columns=[aux], errors="ignore")


# ----------------------------
# 9. Reordenar columnas
# ----------------------------
columnas_finales = [
    "Provincia",
    "Comunidad autónoma",
    "Capital",
    "Ciudad más poblada*",
    "Código postal",
    "Código Ministerio del Interior",
    "Superficie (km²)",
    "Longitud de costa (km)[2]​[3]​",
    "Población",
    "Densidad (hab./km²)",
    "Presupuesto (€)",
    "Extranjeros totales",
    "% de extranjeros",
    "Porcentaje población",
    "Porcentaje superficie",
    "2016",
    "2017",
    "2018",
    "2019",
    "2020",
    "2021",
    "2022",
    "2023",
    "TCAC 2016-23",
    "Órgano de gobierno y administración",
    "Sede (ciudad)"
]

columnas_finales = [c for c in columnas_finales if c in df.columns]
df = df[columnas_finales]


# ----------------------------
# 10. Guardar final
# ----------------------------
df.to_csv("datos_finales.csv", index=False)

print(df.head())
print("\nShape:", df.shape)
print("\nColumnas:", df.columns.tolist())

     Provincia      Comunidad autónoma   Capital Ciudad más poblada*  \
0     Albacete      Castilla-La Mancha  Albacete            Albacete   
1     Alicante    Comunidad Valenciana  Alicante            Alicante   
2      Almería               Andalucía   Almería             Almería   
3  Araba/Álava              País Vasco   Vitoria             Vitoria   
4     Asturias  Principado de Asturias    Oviedo               Gijón   

   Código postal Código Ministerio del Interior  Superficie (km²)  \
0              2                             AB           14924.0   
1              3                              A            5816.0   
2              4                             AL            8774.0   
3              1                             VI            3037.0   
4             33                              O           10603.0   

   Longitud de costa (km)[2]​[3]​  Población  Densidad (hab./km²)  \
0                             NaN     390751                26.18   
1             

In [ ]:
print(df[["Porcentaje población", "Porcentaje superficie"]].head())

   Porcentaje población  Porcentaje superficie
0                   NaN                    NaN
1                   NaN                    NaN
2                   NaN                    NaN
3                   NaN                    NaN
4                   NaN                    NaN


In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("datos_raw.csv")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)


# ----------------------------
# 0. Normalizar nombres de columnas
# ----------------------------
def limpiar_nombre_columna(c):
    c = str(c)
    c = c.replace("\u200b", "")   # zero-width space
    c = c.replace("\xa0", " ")    # non-breaking space
    c = re.sub(r"\s+", " ", c)
    return c.strip()

df.columns = [limpiar_nombre_columna(c) for c in df.columns]


# ----------------------------
# 1. Normalizar provincia
# ----------------------------
def normalizar_provincia(x):
    if pd.isna(x):
        return x

    x = str(x).strip()

    equivalencias = {
        "Coruña, La": "La Coruña",
        "Palmas, Las": "Las Palmas",
        "Rioja, La": "La Rioja",
        "Baleares": "Islas Baleares",
        "Gerona": "Girona",
        "Lérida": "Lleida",
        "Orense": "Ourense",
        "Álava": "Araba/Álava",
        "Guipúzcoa": "Gipuzkoa",
        "Vizcaya": "Bizkaia"
    }

    return equivalencias.get(x, x)


df["Provincia"] = df["Provincia"].astype(str).str.strip().apply(normalizar_provincia)

filas_basura = ["España", "Total", "Otras (Chafarinas y Vélez de la Gomera)", "-", "nan"]
df = df[~df["Provincia"].isin(filas_basura)].copy()


# ----------------------------
# 2. Eliminar columnas claramente basura
# ----------------------------
cols_drop = [
    c for c in df.columns
    if c.startswith("Unnamed")
    or c.startswith("Escudo")
    or c.startswith("Mapa")
    or c.startswith("Posición")
    or c.startswith("Puesto")
    or c.startswith("Provincia.")
]

df = df.drop(columns=cols_drop, errors="ignore")


# ----------------------------
# 3. Unificar columnas equivalentes
# ----------------------------
def unificar_columnas(df, nombre_base):
    cols = [c for c in df.columns if c == nombre_base or c.startswith(nombre_base + "_")]
    if not cols:
        return df

    df[nombre_base] = df[cols].bfill(axis=1).iloc[:, 0]
    cols_a_borrar = [c for c in cols if c != nombre_base]
    df = df.drop(columns=cols_a_borrar, errors="ignore")
    return df


bases_a_unificar = [
    "Comunidad autónoma",
    "Capital",
    "Ciudad más poblada*",
    "Población",
    "Presupuesto (€)",
    "Densidad (hab./km²)",
    "Superficie (km²)",
    "Código postal",
    "Código Ministerio del Interior",
    "Órgano de gobierno y administración",
    "Sede (ciudad)",
    "Extranjeros totales",
    "% de extranjeros",
    "2016",
    "2017",
    "2018",
    "2019",
    "2020",
    "2021",
    "2022",
    "2023",
    "TCAC 2016-23",
    "Porcentaje población",
    "Porcentaje superficie",
    "Longitud de costa (km)[2]​[3]​",
    "Superficie (km²)[2]​"
]

for base in bases_a_unificar:
    df = unificar_columnas(df, base)

# refuerzo extra para porcentajes, por si venían con sufijos raros
for base in ["Porcentaje población", "Porcentaje superficie"]:
    cols = [c for c in df.columns if c == base or c.startswith(base + "_")]
    if cols:
        df[base] = df[cols].bfill(axis=1).iloc[:, 0]


# ----------------------------
# 4. Consolidar duplicados por provincia
# ----------------------------
def first_non_null(series):
    s = series.dropna()
    return s.iloc[0] if not s.empty else np.nan

df = df.groupby("Provincia", as_index=False).agg(first_non_null)


# ----------------------------
# 5. Limpieza de texto
# ----------------------------
def limpiar_texto(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = re.sub(r"\[\d+\]", "", x)
    x = x.replace("\xa0", " ")
    x = x.replace("\u200b", "")
    x = x.strip()

    if x in ["", "-", "—", "nan", "None"]:
        return np.nan

    return x

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(limpiar_texto)


# ----------------------------
# 6. Funciones numéricas robustas
# ----------------------------
def numero_entero_es(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, (int, np.integer)):
        return int(x)

    if isinstance(x, (float, np.floating)):
        if np.isnan(x):
            return np.nan
        return int(round(x))

    s = limpiar_texto(x)
    if pd.isna(s):
        return np.nan

    s = str(s).replace(" ", "")

    # varios puntos -> miles
    if s.count(".") > 1:
        s = s.replace(".", "")
        return pd.to_numeric(s, errors="coerce")

    # un punto y 3 dígitos detrás -> miles
    if s.count(".") == 1 and "," not in s:
        izq, der = s.split(".")
        if der.isdigit() and len(der) == 3:
            s = izq + der
            return pd.to_numeric(s, errors="coerce")

    val = pd.to_numeric(s, errors="coerce")
    if pd.notna(val):
        return int(round(val))

    return np.nan


def numero_decimal_es(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, (int, np.integer, float, np.floating)):
        return float(x)

    s = limpiar_texto(x)
    if pd.isna(s):
        return np.nan

    s = str(s).replace(" ", "")

    if "," in s and "." in s:
        s = s.replace(".", "")
        s = s.replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")

    return pd.to_numeric(s, errors="coerce")


def porcentaje_extranjeros_corregido(x):
    """
    Corrige casos como:
    74  -> 7.4
    218 -> 21.8
    230 -> 23.0
    """
    val = numero_decimal_es(x)
    if pd.isna(val):
        return np.nan

    if 100 < val < 1000:
        return val / 10

    return val


# ----------------------------
# 7. Convertir columnas por tipo
# ----------------------------
cols_enteras = [
    "Código postal",
    "Superficie (km²)",
    "Longitud de costa (km)[2]​[3]​",
    "Población",
    "Presupuesto (€)",
    "Extranjeros totales",
    "2016",
    "2017",
    "2018",
    "2019",
    "2020",
    "2021",
    "2022",
    "2023"
]

for col in cols_enteras:
    if col in df.columns:
        df[col] = df[col].apply(numero_entero_es)

cols_decimales = [
    "Densidad (hab./km²)",
    "TCAC 2016-23",
    "Porcentaje población",
    "Porcentaje superficie"
]

for col in cols_decimales:
    if col in df.columns:
        df[col] = df[col].apply(numero_decimal_es)

if "% de extranjeros" in df.columns:
    df["% de extranjeros"] = df["% de extranjeros"].apply(porcentaje_extranjeros_corregido)


# ----------------------------
# 8. Si queda una superficie auxiliar, usarla para rellenar
# ----------------------------
col_aux = [c for c in df.columns if c.startswith("Superficie (km²)[2]")]
if col_aux:
    aux = col_aux[0]
    df[aux] = df[aux].apply(numero_entero_es)

    if "Superficie (km²)" not in df.columns:
        df["Superficie (km²)"] = np.nan

    df["Superficie (km²)"] = df["Superficie (km²)"].fillna(df[aux])
    df = df.drop(columns=[aux], errors="ignore")


# ----------------------------
# 9. Normalizar comunidades autónomas
# ----------------------------
def normalizar_ccaa(x):
    if pd.isna(x):
        return x

    x = str(x).strip()

    mapa = {
        "Principado de Asturias": "Asturias",
        "Comunidad Foral de Navarra": "Navarra"
    }

    return mapa.get(x, x)

if "Comunidad autónoma" in df.columns:
    df["Comunidad autónoma"] = df["Comunidad autónoma"].apply(normalizar_ccaa)


# ----------------------------
# 10. Reordenar columnas
# ----------------------------
columnas_finales = [
    "Provincia",
    "Comunidad autónoma",
    "Capital",
    "Ciudad más poblada*",
    "Código postal",
    "Código Ministerio del Interior",
    "Superficie (km²)",
    "Longitud de costa (km)[2]​[3]​",
    "Población",
    "Densidad (hab./km²)",
    "Presupuesto (€)",
    "Extranjeros totales",
    "% de extranjeros",
    "Porcentaje población",
    "Porcentaje superficie",
    "2016",
    "2017",
    "2018",
    "2019",
    "2020",
    "2021",
    "2022",
    "2023",
    "TCAC 2016-23",
    "Órgano de gobierno y administración",
    "Sede (ciudad)"
]

columnas_finales = [c for c in columnas_finales if c in df.columns]
df = df[columnas_finales]


# ----------------------------
# 11. Guardar final
# ----------------------------
df.to_csv("datos_finales.csv", index=False)

print(df.head())
print("\nShape:", df.shape)
print("\nColumnas:", df.columns.tolist())
print("\nComprobación porcentajes:")
if "% de extranjeros" in df.columns:
    print(df[["Provincia", "% de extranjeros"]].head(10))
if "Porcentaje población" in df.columns and "Porcentaje superficie" in df.columns:
    print(df[["Provincia", "Porcentaje población", "Porcentaje superficie"]].head(10))

     Provincia    Comunidad autónoma   Capital Ciudad más poblada*  \
0     Albacete    Castilla-La Mancha  Albacete            Albacete   
1     Alicante  Comunidad Valenciana  Alicante            Alicante   
2      Almería             Andalucía   Almería             Almería   
3  Araba/Álava            País Vasco   Vitoria             Vitoria   
4     Asturias              Asturias    Oviedo               Gijón   

   Código postal Código Ministerio del Interior  Superficie (km²)  Población  \
0              2                             AB           14924.0     390751   
1              3                              A            5816.0    2033566   
2              4                             AL            8774.0     770554   
3              1                             VI            3037.0     341961   
4             33                              O           10603.0    1015128   

   Densidad (hab./km²)  Presupuesto (€)  Extranjeros totales  \
0                26.18      179014

# Este es el que funciona bien

In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("datos_raw.csv")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)


# ----------------------------
# 0. Normalizar nombres de columnas
# ----------------------------
def limpiar_nombre_columna(c):
    c = str(c)
    c = c.replace("\u200b", "")
    c = c.replace("\xa0", " ")
    c = re.sub(r"\s+", " ", c)
    return c.strip()

df.columns = [limpiar_nombre_columna(c) for c in df.columns]


# ----------------------------
# 1. Normalizar provincia
# ----------------------------
def normalizar_provincia(x):
    if pd.isna(x):
        return x

    x = str(x).strip()

    equivalencias = {
        "Coruña, La": "La Coruña",
        "Palmas, Las": "Las Palmas",
        "Rioja, La": "La Rioja",
        "Baleares": "Islas Baleares",
        "Gerona": "Girona",
        "Lérida": "Lleida",
        "Orense": "Ourense",
        "Álava": "Araba/Álava",
        "Guipúzcoa": "Gipuzkoa",
        "Vizcaya": "Bizkaia"
    }

    return equivalencias.get(x, x)

df["Provincia"] = df["Provincia"].astype(str).str.strip().apply(normalizar_provincia)

filas_basura = ["España", "Total", "Otras (Chafarinas y Vélez de la Gomera)", "-", "nan"]
df = df[~df["Provincia"].isin(filas_basura)].copy()


# ----------------------------
# 2. Eliminar columnas basura
# ----------------------------
cols_drop = [
    c for c in df.columns
    if c.startswith("Unnamed")
    or c.startswith("Escudo")
    or c.startswith("Mapa")
    or c.startswith("Posición")
    or c.startswith("Puesto")
    or c.startswith("Provincia.")
]

df = df.drop(columns=cols_drop, errors="ignore")


# ----------------------------
# 3. Unificar columnas equivalentes
# ----------------------------
def unificar_columnas(df, nombre_base):
    cols = [c for c in df.columns if c == nombre_base or c.startswith(nombre_base + "_")]
    if not cols:
        return df

    df[nombre_base] = df[cols].bfill(axis=1).iloc[:, 0]
    cols_a_borrar = [c for c in cols if c != nombre_base]
    df = df.drop(columns=cols_a_borrar, errors="ignore")
    return df


bases_a_unificar = [
    "Comunidad autónoma",
    "Capital",
    "Ciudad más poblada*",
    "Población",
    "Presupuesto (€)",
    "Densidad (hab./km²)",
    "Superficie (km²)",
    "Código postal",
    "Código Ministerio del Interior",
    "Órgano de gobierno y administración",
    "Sede (ciudad)",
    "Extranjeros totales",
    "% de extranjeros",
    "2016",
    "2017",
    "2018",
    "2019",
    "2020",
    "2021",
    "2022",
    "2023",
    "TCAC 2016-23",
    "Porcentaje población",
    "Porcentaje superficie",
    "Longitud de costa (km)[2]​[3]​",
    "Superficie (km²)[2]​"
]

for base in bases_a_unificar:
    df = unificar_columnas(df, base)


# ----------------------------
# 4. Consolidar duplicados por provincia
# ----------------------------
def first_non_null(series):
    s = series.dropna()
    return s.iloc[0] if not s.empty else np.nan

df = df.groupby("Provincia", as_index=False).agg(first_non_null)


# ----------------------------
# 5. Limpieza de texto
# ----------------------------
def limpiar_texto(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = re.sub(r"\[\d+\]", "", x)
    x = x.replace("\xa0", " ")
    x = x.replace("\u200b", "")
    x = x.strip()

    if x in ["", "-", "—", "nan", "None"]:
        return np.nan

    return x

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(limpiar_texto)


# ----------------------------
# 6. Funciones numéricas
# ----------------------------
def numero_entero_es(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, (int, np.integer)):
        return int(x)

    if isinstance(x, (float, np.floating)):
        if np.isnan(x):
            return np.nan
        return int(round(x))

    s = limpiar_texto(x)
    if pd.isna(s):
        return np.nan

    s = str(s).replace(" ", "")
    s = s.replace("%", "")

    if s.count(".") > 1:
        s = s.replace(".", "")
        return pd.to_numeric(s, errors="coerce")

    if s.count(".") == 1 and "," not in s:
        izq, der = s.split(".")
        if der.isdigit() and len(der) == 3:
            s = izq + der
            return pd.to_numeric(s, errors="coerce")

    val = pd.to_numeric(s, errors="coerce")
    if pd.notna(val):
        return int(round(val))

    return np.nan


def numero_decimal_es(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, (int, np.integer, float, np.floating)):
        return float(x)

    s = limpiar_texto(x)
    if pd.isna(s):
        return np.nan

    s = str(s).replace(" ", "")
    s = s.replace("%", "")

    if "," in s and "." in s:
        s = s.replace(".", "")
        s = s.replace(",", ".")
    elif "," in s:
        s = s.replace(",", ".")

    return pd.to_numeric(s, errors="coerce")


def porcentaje_extranjeros_corregido(x):
    """
    En esta tabla concreta el scraper deja:
    74 -> 7.4
    218 -> 21.8
    230 -> 23.0
    """
    val = numero_decimal_es(x)
    if pd.isna(val):
        return np.nan
    return val / 10


# ----------------------------
# 7. Convertir columnas por tipo
# ----------------------------
cols_enteras = [
    "Código postal",
    "Superficie (km²)",
    "Longitud de costa (km)[2]​[3]​",
    "Población",
    "Presupuesto (€)",
    "Extranjeros totales",
    "2016",
    "2017",
    "2018",
    "2019",
    "2020",
    "2021",
    "2022",
    "2023"
]

for col in cols_enteras:
    if col in df.columns:
        df[col] = df[col].apply(numero_entero_es)

cols_decimales = [
    "Densidad (hab./km²)",
    "TCAC 2016-23",
    "Porcentaje población",
    "Porcentaje superficie"
]

for col in cols_decimales:
    if col in df.columns:
        df[col] = df[col].apply(numero_decimal_es)

if "% de extranjeros" in df.columns:
    df["% de extranjeros"] = df["% de extranjeros"].apply(porcentaje_extranjeros_corregido)


# ----------------------------
# 8. Rellenar superficie auxiliar
# ----------------------------
col_aux = [c for c in df.columns if c.startswith("Superficie (km²)[2]")]
if col_aux:
    aux = col_aux[0]
    df[aux] = df[aux].apply(numero_entero_es)

    if "Superficie (km²)" not in df.columns:
        df["Superficie (km²)"] = np.nan

    df["Superficie (km²)"] = df["Superficie (km²)"].fillna(df[aux])
    df = df.drop(columns=[aux], errors="ignore")


# ----------------------------
# 9. Normalizar comunidades autónomas
# ----------------------------
def normalizar_ccaa(x):
    if pd.isna(x):
        return x

    x = str(x).strip()

    mapa = {
        "Principado de Asturias": "Asturias",
        "Comunidad Foral de Navarra": "Navarra"
    }

    return mapa.get(x, x)

if "Comunidad autónoma" in df.columns:
    df["Comunidad autónoma"] = df["Comunidad autónoma"].apply(normalizar_ccaa)


# ----------------------------
# 10. Reordenar columnas
# ----------------------------
columnas_finales = [
    "Provincia",
    "Comunidad autónoma",
    "Capital",
    "Ciudad más poblada*",
    "Código postal",
    "Código Ministerio del Interior",
    "Superficie (km²)",
    "Longitud de costa (km)[2]​[3]​",
    "Población",
    "Densidad (hab./km²)",
    "Presupuesto (€)",
    "Extranjeros totales",
    "% de extranjeros",
    "Porcentaje población",
    "Porcentaje superficie",
    "2016",
    "2017",
    "2018",
    "2019",
    "2020",
    "2021",
    "2022",
    "2023",
    "TCAC 2016-23",
    "Órgano de gobierno y administración",
    "Sede (ciudad)"
]

columnas_finales = [c for c in columnas_finales if c in df.columns]
df = df[columnas_finales]


# ----------------------------
# 11. Guardar
# ----------------------------
df.to_csv("datos_finales.csv", index=False)

print(df.head())
print("\nShape:", df.shape)
print("\nColumnas:", df.columns.tolist())
print("\nComprobación:")
print(df[["Provincia", "% de extranjeros", "Porcentaje población", "Porcentaje superficie"]].head(10))

     Provincia    Comunidad autónoma   Capital Ciudad más poblada*  \
0     Albacete    Castilla-La Mancha  Albacete            Albacete   
1     Alicante  Comunidad Valenciana  Alicante            Alicante   
2      Almería             Andalucía   Almería             Almería   
3  Araba/Álava            País Vasco   Vitoria             Vitoria   
4     Asturias              Asturias    Oviedo               Gijón   

   Código postal Código Ministerio del Interior  Superficie (km²)  Población  \
0              2                             AB           14924.0     390751   
1              3                              A            5816.0    2033566   
2              4                             AL            8774.0     770554   
3              1                             VI            3037.0     341961   
4             33                              O           10603.0    1015128   

   Densidad (hab./km²)  Presupuesto (€)  Extranjeros totales  \
0                26.18      179014